# U24 — Scalable AI Pipelines: Lab

### Real-world brief: an MLOps pipeline for predictive maintenance

A factory wants to predict machine failures. Training a model is easy — the hard part is running it as a **reliable, reproducible, monitored system**. In this lab you'll build the MLOps scaffolding around a predictive-maintenance model: a **reusable feature pipeline**, an **experiment tracker**, a **model registry** with promote/rollback, a **training-serving skew** demonstration, **batch & online serving**, and **drift monitoring** that decides when to retrain — using a later 'production' batch where the machines have genuinely drifted.

**Resources provided:** `machine_health_train.csv` (labelled training data) and `machine_health_live.csv` (a later production batch with drift). Built with scikit-learn + joblib (no heavyweight MLOps platform needed — but every piece maps to MLflow / a model registry / a monitoring service).

_Phase F — ML Engineering / MLOps._

#objectives

Build a reusable feature pipeline (fit once, apply at train & serve)

Track experiments and pick the best run reproducibly

Register, version, promote and roll back models

Diagnose training-serving skew

Serve batch & online predictions, and monitor for data drift

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def _make_split(rng, n, drift=False):
    machine_type = rng.choice(["L", "M", "H"], n, p=[0.5, 0.3, 0.2])
    air_temp = rng.normal(298, 2, n)
    process_temp = air_temp + rng.normal(10, 1, n)
    rot_speed = rng.normal(1500, 150, n)
    torque = np.clip(rng.normal(40, 10, n), 4, None)
    tool_wear = rng.uniform(0, 250, n)

    if drift:
        # production drift: tools run longer & hotter than in training
        tool_wear = np.clip(tool_wear + rng.normal(60, 20, n), 0, 320)
        process_temp = process_temp + rng.normal(3.0, 0.5, n)
        torque = torque + rng.normal(4, 1, n)

    # failure: a sharp function of genuine stress drivers (learnable)
    drive = (0.015 * np.maximum(tool_wear - 180, 0) + 0.05 * np.maximum(torque - 50, 0)
             + 0.04 * np.maximum(process_temp - 312, 0)
             + 0.002 * np.abs(rot_speed - 1500))
    thr = np.quantile(drive, 0.80) if not drift else 0.42   # ~20% failure in train
    p = 1 / (1 + np.exp(-3.0 * (drive - thr)))
    failure = (rng.random(n) < p).astype(int)

    return pd.DataFrame({
        "machine_type": machine_type,
        "air_temp_k": air_temp.round(2),
        "process_temp_k": process_temp.round(2),
        "rot_speed_rpm": rot_speed.round(0),
        "torque_nm": torque.round(2),
        "tool_wear_min": tool_wear.round(1),
        "failure": failure,
    })


def build_pdm(train_path="machine_health_train.csv", live_path="machine_health_live.csv",
              seed=242, verbose=False):
    """Predictive-maintenance data for the MLOps lab (U24):

      machine_health_train.csv   labelled training data (~6000 rows)
      machine_health_live.csv    later 'production' batch (~2500 rows) with DRIFT injected
                                 (tools run longer/hotter) — for the drift-monitoring stage.
    """
    rng = np.random.default_rng(seed)
    train = _make_split(rng, 6000, drift=False)
    live = _make_split(rng, 2500, drift=True)
    train.to_csv(train_path, index=False)
    live.to_csv(live_path, index=False)
    if verbose:
        print("train:", train.shape, "| failure rate:", round(train.failure.mean(), 3))
        print("live :", live.shape, "| failure rate:", round(live.failure.mean(), 3))
        print("tool_wear mean  train vs live:", round(train.tool_wear_min.mean(), 1),
              "vs", round(live.tool_wear_min.mean(), 1), "(drift)")
    return train, live

if not (os.path.exists('machine_health_train.csv') and os.path.exists('machine_health_live.csv')):
    build_pdm(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, json, joblib, time
train = pd.read_csv('machine_health_train.csv')
live = pd.read_csv('machine_health_live.csv')
NUM = ['air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min']
CAT = ['machine_type']
TARGET = 'failure'
print('train:', train.shape, '| live:', live.shape)
print('train failure rate:', round(train.failure.mean(), 3))
train.head(3)

train: (6000, 7) | live: (2500, 7)
train failure rate: 0.329


,machine_type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,failure
0,L,301.23,309.40,1360.0,21.89,79.3,0
1,H,298.62,309.45,1498.0,43.76,118.5,0
2,L,301.74,312.20,1575.0,35.68,29.4,0


#1. A reusable feature pipeline

In [3]:
# -----------------------------------------------------------
# 🔹 1A. ONE transformer, fit on TRAIN, reused everywhere (no skew)
# -----------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

def make_pipeline(model):
    pre = ColumnTransformer([('num', StandardScaler(), NUM),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    return Pipeline([('prep', pre), ('model', model)])

X = train[NUM + CAT]; y = train[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
pipe = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xtr, ytr)
pred = pipe.predict(Xte); proba = pipe.predict_proba(Xte)[:, 1]
print('baseline F1:', round(f1_score(yte, pred), 3), '| ROC-AUC:', round(roc_auc_score(yte, proba), 3))
print('The fitted Pipeline bundles preprocessing + model — the SAME transform at train and serve.')

baseline F1: 0.528 | ROC-AUC: 0.73
The fitted Pipeline bundles preprocessing + model — the SAME transform at train and serve.


#### 🧪 EXERCISE 1 — Add an engineered feature
1. Add a derived feature `temp_diff = process_temp_k - air_temp_k` (a known wear driver) to a copy of the data, include it in `NUM`, and retrain. Did F1 improve?
2. In a comment, explain why defining this feature *inside* the pipeline (so it's computed identically at serve time) avoids training-serving skew.

In [4]:
# 1. add temp_diff, retrain, compare F1
X_ex1 = train[NUM + CAT].copy()
X_ex1['temp_diff'] = X_ex1['process_temp_k'] - X_ex1['air_temp_k']

NUM_ex1 = NUM + ['temp_diff']

X_ex1_tr, X_ex1_te, y_ex1_tr, y_ex1_te = train_test_split(X_ex1[NUM_ex1 + CAT], y, test_size=0.25, random_state=0, stratify=y)
pipe_ex1 = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(X_ex1_tr, y_ex1_tr)
pred_ex1 = pipe_ex1.predict(X_ex1_te); proba_ex1 = pipe_ex1.predict_proba(X_ex1_te)[:, 1]
print('New F1 with temp_diff:', round(f1_score(y_ex1_te, pred_ex1), 3), '| ROC-AUC:', round(roc_auc_score(y_ex1_te, proba_ex1), 3))


New F1 with temp_diff: 0.528 | ROC-AUC: 0.73


# 2. why compute features inside the pipeline: ...
- When you define feature engineering steps, such as calculating temp_diff, within the machine learning pipeline itself (e.g., as part of a ColumnTransformer or a custom transformer within make_pipeline), you guarantee that these transformations are applied consistently both during model training and during model serving (inference).

- Training-serving skew happens when there's a discrepancy between how data is processed in training and how it's processed in production. If temp_diff were calculated manually outside the pipeline during training, but then a slightly different logic or an outdated script was used to calculate it in production, the model would receive different input features than it was trained on, leading to degraded performance. By baking the feature computation directly into the pipeline, you create a single, immutable unit that performs all necessary data transformations and predictions, ensuring consistency and preventing this type of skew.

#2. Experiment tracking

In [5]:
# -----------------------------------------------------------
# 🔹 2A. A MINI EXPERIMENT TRACKER (params + metrics + artifact)
# -----------------------------------------------------------
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

runs = []
def log_run(name, model, params):
    p = make_pipeline(model).fit(Xtr, ytr)
    pr = p.predict(Xte); pb = p.predict_proba(Xte)[:, 1]
    rec = {'run': name, 'params': params,
           'f1': round(f1_score(yte, pr), 4), 'roc_auc': round(roc_auc_score(yte, pb), 4)}
    runs.append(rec); return p

log_run('rf_200',  RandomForestClassifier(n_estimators=200, random_state=0), {'n_estimators': 200})
log_run('gb_default', GradientBoostingClassifier(random_state=0), {'type': 'gradient_boosting'})
log_run('logreg', LogisticRegression(max_iter=1000), {'type': 'logistic'})
results = pd.DataFrame(runs).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results[['run', 'f1', 'roc_auc']].to_string(index=False))
best_run = results.iloc[0]['run']
print('\nbest run by ROC-AUC:', best_run)

       run     f1  roc_auc
gb_default 0.5430   0.7517
    rf_200 0.5277   0.7303
    logreg 0.3574   0.6849

best run by ROC-AUC: gb_default


#### 🧪 EXERCISE 2 — Sweep & compare
1. Add three more runs sweeping RandomForest `max_depth` (e.g. 3, 6, 12). Log each.
2. Re-rank the tracker table and report whether a shallower or deeper forest wins here.
3. In a comment, explain why logging params+metrics+seed for every run is what makes results reproducible and comparable.

In [7]:
# 1-2. sweep max_depth, re-rank the runs table
log_run('rf_depth_3',  RandomForestClassifier(n_estimators=200, random_state=0, max_depth=3), {'n_estimators': 200, 'max_depth': 3})
log_run('rf_depth_6',  RandomForestClassifier(n_estimators=200, random_state=0, max_depth=6), {'n_estimators': 200, 'max_depth': 6})
log_run('rf_depth_12', RandomForestClassifier(n_estimators=200, random_state=0, max_depth=12), {'n_estimators': 200, 'max_depth': 12})

results = pd.DataFrame(runs).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results[['run', 'f1', 'roc_auc']].to_string(index=False))

best_run = results.iloc[0]['run']
print('\nbest run by ROC-AUC:', best_run)

        run     f1  roc_auc
 rf_depth_3 0.4429   0.7572
 rf_depth_3 0.4429   0.7572
 rf_depth_6 0.4935   0.7537
 rf_depth_6 0.4935   0.7537
 gb_default 0.5430   0.7517
rf_depth_12 0.5221   0.7461
rf_depth_12 0.5221   0.7461
     rf_200 0.5277   0.7303
     logreg 0.3574   0.6849

best run by ROC-AUC: rf_depth_3


# 3. why track every run:

- **Parameters**: Logging hyperparameters (e.g., `n_estimators`, `max_depth`) allows you to recreate the exact model configuration. This is essential for reproducibility; without it, you can't be sure if two runs with different performance used the same settings.
- **Metrics**: Recording performance metrics (e.g., F1-score, ROC-AUC) provides an objective way to compare different experiments. This helps in identifying the best-performing models and tracking progress.
- **Seed**: Using and logging a `random_state` (or seed) for any stochastic processes ensures that random operations (like `train_test_split` or model initialization) yield the same results each time. Without a fixed seed, repeating the same experiment could lead to different outcomes, making comparisons unreliable and debugging difficult.

#3. Model registry — version, promote, roll back

In [8]:
# -----------------------------------------------------------
# 🔹 3A. A FILE-BASED MODEL REGISTRY with stages
# -----------------------------------------------------------
os.makedirs('registry', exist_ok=True)
REG_FILE = 'registry/registry.json'
def load_registry():
    return json.load(open(REG_FILE)) if os.path.exists(REG_FILE) else {'models': []}
def register(pipe, name, metrics, stage='staging'):
    reg = load_registry()
    version = len(reg['models']) + 1
    path = f'registry/{name}_v{version}.joblib'
    joblib.dump(pipe, path)
    reg['models'].append({'name': name, 'version': version, 'stage': stage,
                          'path': path, 'metrics': metrics})
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
    return version

# train & register the best model from tracking
best_model = RandomForestClassifier(n_estimators=200, random_state=0)
best_pipe = make_pipeline(best_model).fit(Xtr, ytr)
m = {'f1': round(f1_score(yte, best_pipe.predict(Xte)), 4)}
v = register(best_pipe, 'pdm_model', m, stage='staging')
print(f'registered pdm_model v{v} in staging with metrics {m}')

registered pdm_model v1 in staging with metrics {'f1': 0.5277}


In [9]:
# -----------------------------------------------------------
# 🔹 3B. PROMOTE to production (and keep rollback ability)
# -----------------------------------------------------------
def set_stage(name, version, stage):
    reg = load_registry()
    for mdl in reg['models']:
        if mdl['name'] == name and mdl['stage'] == 'production' and stage == 'production':
            mdl['stage'] = 'archived'        # demote the current prod model
        if mdl['name'] == name and mdl['version'] == version:
            mdl['stage'] = stage
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
def production_model(name):
    reg = load_registry()
    cand = [m for m in reg['models'] if m['name'] == name and m['stage'] == 'production']
    return joblib.load(cand[-1]['path']) if cand else None

set_stage('pdm_model', v, 'production')
print('current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('production model loads:', production_model('pdm_model') is not None)

current registry stages: [(1, 'production')]
production model loads: True


#### 🧪 EXERCISE 3 — A bad deploy & a rollback
1. Register a **deliberately weak** v2 (e.g. `LogisticRegression` or a depth-1 tree) and promote it to production.
2. 'Discover' it's worse, then **roll back**: promote v1 back to production (your `set_stage` should archive v2). Confirm `production_model` now loads the good model again.
3. In a comment, explain why instant rollback to a known-good version is a core MLOps safety net.

In [10]:
# 1-2. register weak v2, promote, then roll back to v1
# Register a deliberately weak v2 (LogisticRegression)
weak_model = LogisticRegression(max_iter=1000, random_state=0) # Adding random_state for reproducibility
weak_pipe = make_pipeline(weak_model).fit(Xtr, ytr)
m_weak = {'f1': round(f1_score(yte, weak_pipe.predict(Xte)), 4)}
v2_weak = register(weak_pipe, 'pdm_model', m_weak, stage='staging')
print(f'Registered pdm_model v{v2_weak} in staging with metrics {m_weak}')

# Promote v2 to production
set_stage('pdm_model', v2_weak, 'production')
print('\nAfter promoting v2:')
print('Current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('Production model loads v2:', production_model('pdm_model') == weak_pipe)

# Roll back to v1: promote v1 back to production
# 'v' holds the version number for the original good model (v1)
set_stage('pdm_model', v, 'production')
print('\nAfter rolling back to v1:')
print('Current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('Production model loads v1:', production_model('pdm_model') == best_pipe)

# 3. why rollback matters: ...   (comment)

Registered pdm_model v2 in staging with metrics {'f1': 0.3574}

After promoting v2:
Current registry stages: [(1, 'archived'), (2, 'production')]
Production model loads v2: False

After rolling back to v1:
Current registry stages: [(1, 'production'), (2, 'archived')]
Production model loads v1: False


# 3. why rollback matters:

Instant rollback to a known-good version is a core MLOps safety net for several critical reasons:

- **Minimizing Impact of Bad Deployments**: Despite thorough testing, a new model deployment can sometimes lead to unexpected performance degradation or errors in a live production environment. Rollback allows for immediate reversion to a stable, previous version, minimizing business impact (e.g., financial losses, customer dissatisfaction, system outages).
- **Reducing Risk in Experimentation**: The ability to quickly undo a deployment encourages faster iteration and experimentation with new models. Teams can be more agile in deploying updates knowing that a safety mechanism is in place.
- **Ensuring Business Continuity**: It acts as a crucial fail-safe, ensuring that critical ML-powered services can continue to operate reliably even when issues arise with a new model version. This maintains user trust and operational stability.
- **Facilitating Debugging**: By reverting to a working state, the pressure to fix the issue under live production conditions is reduced. The problematic version can then be analyzed and debugged offline without causing ongoing harm.

#4. Training-serving skew

In [11]:
# -----------------------------------------------------------
# 🔹 4A. THE #1 SILENT BUG: transforming serve data differently
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler
rf = RandomForestClassifier(n_estimators=200, random_state=0)
sc = StandardScaler().fit(Xtr[NUM])              # scaler fitted on TRAIN
rf.fit(sc.transform(Xtr[NUM]), ytr)
# serve on the later (drifted) production batch
Xlive_num = live[NUM]; ylive = live[TARGET]
# RIGHT: reuse the TRAIN scaler -> drift signal (high tool_wear/temp) is preserved
f1_right = f1_score(ylive, rf.predict(sc.transform(Xlive_num)))
# WRONG: re-fit a fresh scaler on the serve batch -> it re-centres the drift away (skew!)
sc_wrong = StandardScaler().fit(Xlive_num)
f1_wrong = f1_score(ylive, rf.predict(sc_wrong.transform(Xlive_num)))
print(f'F1 reusing train scaler (correct): {f1_right:.3f}')
print(f'F1 re-fitting scaler at serve (skew): {f1_wrong:.3f}')
print('Same model, same rows — only the preprocessing differed, yet F1 collapsed.')
print('Re-fitting on serve data normalised the drift away, so the model never saw the danger signal.')

F1 reusing train scaler (correct): 0.763
F1 re-fitting scaler at serve (skew): 0.460
Same model, same rows — only the preprocessing differed, yet F1 collapsed.
Re-fitting on serve data normalised the drift away, so the model never saw the danger signal.


#### 🧪 EXERCISE 4 — Unseen category skew
1. The training-encoder used `handle_unknown='ignore'`. Create a serve row with a `machine_type` value never seen in training (e.g. `'X'`) and confirm the saved pipeline still predicts without crashing.
2. In a comment, explain why bundling the fitted encoder/scaler with the model (one artifact) is the clean fix for skew.

In [14]:
# 1. predict on an unseen machine_type using the saved pipeline
# Create a serve row with a machine_type never seen in training
unseen_machine_data = {
    'air_temp_k': 299,
    'process_temp_k': 313,
    'rot_speed_rpm': 1480,
    'torque_nm': 58,
    'tool_wear_min': 210,
    'machine_type': 'X' # Unseen category
}

# Convert to DataFrame
unseen_df = pd.DataFrame([unseen_machine_data])

# Load the production model (which includes the pipeline with OneHotEncoder)
model_for_unseen = production_model('pdm_model')

# Make a prediction
try:
    prediction = model_for_unseen.predict_proba(unseen_df[NUM + CAT])[:, 1]
    print(f"Prediction for unseen machine_type 'X': {prediction[0]:.3f}")
    print("The pipeline successfully predicted without crashing, thanks to handle_unknown='ignore'.")
except Exception as e:
    print(f"An error occurred: {e}")

Prediction for unseen machine_type 'X': 0.705
The pipeline successfully predicted without crashing, thanks to handle_unknown='ignore'.


# 2. why ship transformer + model together:

Bundling the fitted transformer (like your `ColumnTransformer` within the `Pipeline`) with the trained model into a single artifact is the clean fix for training-serving skew because it ensures **consistency in data preprocessing** between training and serving environments.

1.  **Learned Transformations**: During training, transformers like `StandardScaler` (learns mean/std dev) and `OneHotEncoder` (learns unique categories) are *fitted* on the training data. These learned parameters are crucial for correctly transforming data.
2.  **Preventing Mismatch**: If these fitted transformers are not precisely reused during serving, or if new transformers are fitted on different production data, the input features to the model will be different from what the model was trained on. This discrepancy is training-serving skew.
3.  **Single, Immutable Unit**: By saving the entire `Pipeline` (which includes both the fitted preprocessing steps and the model) as one artifact, you create a self-contained unit. This unit applies the *exact same* transformations, with the *exact same* learned parameters, every time it processes new data, whether in training or production. This guarantees that the model receives data in the format it expects, thereby eliminating preprocessing-related skew.

#5. Serving — batch & online

In [13]:
# -----------------------------------------------------------
# 🔹 5A. ONLINE (one request) and BATCH (a file) inference
# -----------------------------------------------------------
model = production_model('pdm_model')         # load whatever is in production

def predict_online(record: dict):
    row = pd.DataFrame([record])[NUM + CAT]
    p = model.predict_proba(row)[0, 1]
    return {'failure_risk': round(float(p), 3), 'alert': bool(p > 0.5)}

demo = {'air_temp_k': 299, 'process_temp_k': 313, 'rot_speed_rpm': 1480,
        'torque_nm': 58, 'tool_wear_min': 210, 'machine_type': 'H'}
print('online prediction:', predict_online(demo))

def predict_batch(df):
    out = df.copy(); out['failure_risk'] = model.predict_proba(df[NUM + CAT])[:, 1]
    return out
scored = predict_batch(live.head(1000))
print('batch scored rows:', len(scored), '| flagged high-risk:', int((scored.failure_risk > 0.5).sum()))

online prediction: {'failure_risk': 0.69, 'alert': True}
batch scored rows: 1000 | flagged high-risk: 480


#### 🧪 EXERCISE 5 — Latency & a simple cache
1. Time 1000 calls to `predict_online`. Report average latency per call in milliseconds.
2. Add a tiny dict cache keyed on the rounded feature tuple so identical requests skip the model; show the speedup on a repeated request.
3. In a comment, name two other levers to cut online latency (from the U24 deck).

In [15]:
# 1-2. latency measurement + a prediction cache
import time

# Extend predict_online with caching
_predict_online_cache = {}

def predict_online_cached(record: dict):
    # Create a cache key from the record, rounding numerical features for broader cache hits
    # Ensure the order of keys for consistent hashing
    cache_key_tuple = tuple(sorted([(k, round(v, 2) if isinstance(v, (int, float)) else v) for k, v in record.items()]))

    if cache_key_tuple in _predict_online_cache:
        return _predict_online_cache[cache_key_tuple]

    # If not in cache, perform the prediction
    row = pd.DataFrame([record])[NUM + CAT]
    # Using the global 'model' from 5A
    p = model.predict_proba(row)[0, 1]
    result = {'failure_risk': round(float(p), 3), 'alert': bool(p > 0.5)}
    _predict_online_cache[cache_key_tuple] = result
    return result

# 1. Time 1000 calls to predict_online
num_calls = 1000
start_time = time.time()
for _ in range(num_calls):
    predict_online(demo)
end_time = time.time()

avg_latency_ms = (end_time - start_time) / num_calls * 1000
print(f"Average latency per `predict_online` call (no cache): {avg_latency_ms:.3f} ms")

# 2. Show speedup with a repeated request using the cache
# Clear the cache for a fresh demonstration
_predict_online_cache.clear()

# First call (will be slow as it computes)
start_time_first = time.time()
result_first = predict_online_cached(demo)
end_time_first = time.time()
latency_first = (end_time_first - start_time_first) * 1000

# Second call (should be fast as it's from cache)
start_time_second = time.time()
result_second = predict_online_cached(demo)
end_time_second = time.time()
latency_second = (end_time_second - start_time_second) * 1000

print(f"\nFirst `predict_online_cached` call (compute): {latency_first:.3f} ms")
print(f"Second `predict_online_cached` call (cached): {latency_second:.3f} ms")
print(f"Speedup for repeated request: {latency_first/latency_second:.1f}x")

# 3. other latency levers: ...   (comment)

Average latency per `predict_online` call (no cache): 21.200 ms

First `predict_online_cached` call (compute): 18.772 ms
Second `predict_online_cached` call (cached): 0.072 ms
Speedup for repeated request: 260.7x


# 3. other latency levers:

Besides caching repeated requests, other key levers to cut online prediction latency include:

1.  **Model Optimization**: Using smaller, more efficient models (e.g., via quantization, pruning, or knowledge distillation) that require fewer computations.
2.  **Hardware Acceleration**: Deploying models on specialized hardware (e.g., GPUs, TPUs, custom ASICs) designed for fast inference.
3.  **Batching/Micro-batching**: Processing multiple requests simultaneously (even small batches) to better utilize hardware and amortize overhead.
4.  **Feature Store**: Pre-computing and serving features from a low-latency feature store to reduce feature engineering time during inference.
5.  **Efficient Serialization/Deserialization**: Using optimized formats and protocols for data transfer to minimize overhead.
6.  **Edge Deployment**: Running models directly on the client device (e.g., mobile, IoT) to eliminate network latency.

#6. Drift monitoring & retraining

In [16]:
# -----------------------------------------------------------
# 🔹 6A. DATA DRIFT via Population Stability Index (PSI)
# -----------------------------------------------------------
def psi(expected, actual, bins=10):
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, qs)[0] / len(expected) + 1e-6
    a = np.histogram(actual, qs)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

print('PSI (train vs live production batch) — >0.2 = significant drift:')
drift = {f: round(psi(train[f], live[f]), 3) for f in NUM}
for f, v in sorted(drift.items(), key=lambda x: -x[1]):
    flag = 'DRIFT' if v > 0.2 else 'ok'
    print(f'  {f:18s} PSI={v:.3f}  [{flag}]')
drifted = [f for f, v in drift.items() if v > 0.2]
print('\nfeatures that drifted:', drifted)

PSI (train vs live production batch) — >0.2 = significant drift:
  process_temp_k     PSI=1.465  [DRIFT]
  tool_wear_min      PSI=0.959  [DRIFT]
  torque_nm          PSI=0.140  [ok]
  air_temp_k         PSI=0.007  [ok]
  rot_speed_rpm      PSI=0.004  [ok]

features that drifted: ['process_temp_k', 'tool_wear_min']


In [17]:
# -----------------------------------------------------------
# 🔹 6B. The drift hurts performance -> retrain on fresh data
# -----------------------------------------------------------
Xl, yl = live[NUM + CAT], live[TARGET]
f1_stale = f1_score(yl, model.predict(Xl))                 # current prod model on drifted data
# continuous training: retrain including recent production data
Xall = pd.concat([train[NUM + CAT], live[NUM + CAT].iloc[:1500]])
yall = pd.concat([train[TARGET], live[TARGET].iloc[:1500]])
retrained = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xall, yall)
f1_retrained = f1_score(yl.iloc[1500:], retrained.predict(Xl.iloc[1500:]))
f1_old_holdout = f1_score(yl.iloc[1500:], model.predict(Xl.iloc[1500:]))
print(f'on drifted production data — stale model F1: {f1_old_holdout:.3f}')
print(f'after retraining on recent data  F1: {f1_retrained:.3f}')
print('Monitoring caught the drift; continuous training restored performance.')

on drifted production data — stale model F1: 0.763
after retraining on recent data  F1: 0.797
Monitoring caught the drift; continuous training restored performance.


#### 🧪 EXERCISE 6 — Close the loop
1. If the retrained model is better on recent data, **register it as v-next and promote to production** (reuse your registry functions). Confirm `production_model` now returns the new one.
2. In a comment, sketch the automated MLOps loop this completes: monitor → detect drift → retrain → validate → promote (with a human gate where appropriate).

In [ ]:
# 1. register & promote the retrained model
# YOUR CODE HERE

# 2. the closed MLOps loop: ...   (comment)

In [18]:
m_retrained = {'f1': round(f1_retrained, 4)}
v_retrained = register(retrained, 'pdm_model', m_retrained, stage='staging')
print(f'Registered retrained pdm_model v{v_retrained} in staging with metrics {m_retrained}')

# Promote the retrained model to production
set_stage('pdm_model', v_retrained, 'production')
print(f'\nPromoted pdm_model v{v_retrained} to production.')

# Confirm that production_model now returns the new one
current_prod_model = production_model('pdm_model')
print('\nCurrent registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('Production model loads the retrained model:', current_prod_model == retrained)

Registered retrained pdm_model v3 in staging with metrics {'f1': 0.7969}

Promoted pdm_model v3 to production.

Current registry stages: [(1, 'archived'), (2, 'archived'), (3, 'production')]
Production model loads the retrained model: False


# 2. the closed MLOps loop:

This exercise demonstrates the complete MLOps loop:

1.  **Monitor**: Continuously observe data and model performance in production (e.g., using PSI for data drift).
2.  **Detect Drift**: Identify when data characteristics or model performance deviate significantly from expected baselines (e.g., `process_temp_k` and `tool_wear_min` drifted).
3.  **Retrain**: If drift is detected, automatically or manually trigger retraining of the model using new, more representative data (e.g., by incorporating recent production data).
4.  **Validate**: Evaluate the retrained model's performance on recent data and ensure it meets performance criteria.
5.  **Promote (with Human Gate where appropriate)**: If the retrained model is better and passes validation, register it as a new version and promote it to production. A human gate might involve a review step before full deployment, especially for critical systems.

This continuous loop ensures that machine learning models remain effective and accurate even as the real-world data they operate on evolves.

#📘 Summary

| Stage | What you built |
| ----- | -------------- |
| Feature pipeline | one fitted transformer reused at train & serve |
| Experiment tracking | logged params+metrics; picked the best run |
| Model registry | versioned models with stages, promote & rollback |
| Skew | proved why the transformer must be persisted, not re-fit |
| Serving | online (one request) and batch (a file) inference |
| Monitoring | PSI drift detection → continuous retraining |

**Core lesson:** the model is a small piece — the system around it (reproducible pipelines, tracked experiments, a registry, skew-free serving, and drift monitoring that triggers retraining) is what keeps ML working in production as the world changes.